In [235]:
"""
This notebook creates the TrafficPerTerritory table the first time. Once created and loaded in the data warehouse it will be updated by another
notebook or script that will only retrieve new data from the API
"""

'\nThis notebook creates the TrafficPerTerritory table the first time. Once created and loaded in the data warehouse it will be updated by another\nnotebook or script that will only retrieve new data from the API\n'

In [236]:
import pandas as pd
import utils as u

In [237]:
# url passenger data
url_pas = "https://datos.canarias.es/api/estadisticas/statistical-resources/v1.0/datasets/ISTAC/C00017A_000013/~latest.csv?lang=en"
# url goods and mail data
url_gm = "https://datos.canarias.es/api/estadisticas/statistical-resources/v1.0/datasets/ISTAC/C00017A_000014/~latest.csv?lang=en"
# url operations
url_o = "https://datos.canarias.es/api/estadisticas/statistical-resources/v1.0/datasets/ISTAC/C00017A_000015/~latest.csv?lang=en"

In [238]:
# Last updated passenger data
passg = pd.read_csv(u.get_data_from_API_call(url_pas))
# Last updated goods and mail data
goodsm = pd.read_csv(u.get_data_from_API_call(url_gm))
# Last updated operations data
operat = pd.read_csv(u.get_data_from_API_call(url_o))

In [239]:
# For some reason the operations table changes the name of the column of the stopover airport
operat.rename({'AEROPUERTO_ORIGEN_DESTINO_CODE': 'AEROPUERTO_ESCALA_CODE', 'AEROPUERTO_ORIGEN_DESTINO#en': 'AEROPUERTO_ESCALA#en'}, axis=1, inplace=True)

In [240]:
dfs = [passg, goodsm, operat]

In [241]:
# Delete unnecesary columns
for df in dfs:
    df.drop(columns=df.columns[df.columns.str.endswith("#es")], inplace=True)
    df.drop(columns=['CONFIDENCIALIDAD_OBSERVACION#en', 'ESTADO_OBSERVACION_CODE', 'ESTADO_OBSERVACION#en'], inplace=True)

In [242]:
# Merge passengers and operations
df_f = passg.merge(operat, on=['TERRITORIO_CODE', 'AEROPUERTO_ESCALA_CODE', 'MOVIMIENTO_AERONAVE_CODE', 'SERVICIO_AEREO_CODE', 'TIME_PERIOD_CODE'], suffixes=("", "_op"))

df_f.drop(columns=[col for col in df_f.columns if col.endswith('op') and not col.endswith('_VALUE_op')], inplace=True)

In [243]:
only_goods = goodsm.loc[goodsm['MEDIDAS_CODE'] == "MERCANCIA"].copy(deep=True)

only_mail = goodsm.loc[goodsm['MEDIDAS_CODE'] == "CORREO"].copy(deep=True)

In [244]:
# Merge p&op with goods 
df_f = df_f.merge(only_goods, on=['TERRITORIO_CODE', 'AEROPUERTO_ESCALA_CODE', 'MOVIMIENTO_AERONAVE_CODE', 'SERVICIO_AEREO_CODE', 'TIME_PERIOD_CODE'], suffixes=("", "_goods"))

df_f.drop(columns=[col for col in df_f.columns if col.endswith('goods') and col not in ['OBS_VALUE_goods']], inplace=True)

# Merge p&op&goods with mail 
df_f = df_f.merge(only_mail, on=['TERRITORIO_CODE', 'AEROPUERTO_ESCALA_CODE', 'MOVIMIENTO_AERONAVE_CODE', 'SERVICIO_AEREO_CODE', 'TIME_PERIOD_CODE'], suffixes=("", "_mail"))

df_f.drop(columns=[col for col in df_f.columns if col.endswith('mail') and col not in ['OBS_VALUE_mail']], inplace=True)

Change columns to Id's, change name of columns

In [245]:
df_f.drop(columns=[col for col in df_f.columns if col.endswith('#en') and col not in ['TIME_PERIOD#en']], inplace=True)
df_f.drop(columns=['TIME_PERIOD_CODE', 'MEDIDAS_CODE'], inplace=True)

In [246]:
territory = pd.read_csv('./csv_tables/Final_Territory.csv')
airservice = pd.read_csv('./csv_tables/Final_AirService.csv')
aircraftmovement = pd.read_csv('./csv_tables/Final_AircraftMovement.csv')

In [247]:
df_f.loc[pd.isna(df_f["AEROPUERTO_ESCALA_CODE"])]

,TERRITORIO_CODE,AEROPUERTO_ESCALA_CODE,MOVIMIENTO_AERONAVE_CODE,SERVICIO_AEREO_CODE,TIME_PERIOD#en,OBS_VALUE,OBS_VALUE_op,OBS_VALUE_goods,OBS_VALUE_mail


In [248]:
# Replace values in df_f["TERRITORIO_CODE"] with corresponding TerritoryId
df_f["TERRITORIO_CODE"] = df_f["TERRITORIO_CODE"].map(dict(zip(territory["TerritoryCode"], territory["TerritoryId"])))

# Replace values in df_f["AEROPUERTO_ESCALA_CODE"] with corresponding TerritoryId
df_f["AEROPUERTO_ESCALA_CODE"] = df_f["AEROPUERTO_ESCALA_CODE"].map(dict(zip(territory["TerritoryCode"], territory["TerritoryId"])))
# Delete rows where AEROPUERTO_ESCALA_CODE hasnt matched (deleted territories like Total)
df_f.dropna(inplace=True)
df_f["AEROPUERTO_ESCALA_CODE"] = df_f["AEROPUERTO_ESCALA_CODE"].astype(int)

df_f["SERVICIO_AEREO_CODE"] = df_f["SERVICIO_AEREO_CODE"].map(dict(zip(airservice['AirServiceCode'], airservice['AirServiceId'])))

df_f["MOVIMIENTO_AERONAVE_CODE"] = df_f["MOVIMIENTO_AERONAVE_CODE"].map(dict(zip(aircraftmovement['AircraftMovementCode'], aircraftmovement['AircraftMovementId'])))

In [249]:
# Delete year only dates
df_f = df_f.loc[~df_f['TIME_PERIOD#en'].str.match(r'^20[0-9][0-9]$', na=False)]

df_f['TIME_PERIOD#en'] = pd.to_datetime(df_f['TIME_PERIOD#en'], format='%m/%Y')

In [251]:
df_f.sort_values(by='TIME_PERIOD#en', ascending=False)

,TERRITORIO_CODE,AEROPUERTO_ESCALA_CODE,MOVIMIENTO_AERONAVE_CODE,SERVICIO_AEREO_CODE,TIME_PERIOD#en,OBS_VALUE,OBS_VALUE_op,OBS_VALUE_goods,OBS_VALUE_mail
163158,10,10,2,0,2025-07-01,967702.0,16978.0,637491.0,180819.0
13358,1,15,0,3,2025-07-01,15520.0,88.0,818.0,25.0
6078,10,15,2,3,2025-07-01,334267.0,1875.0,25669.0,25.0
84758,4,12,0,2,2025-07-01,95.0,27.0,80505.0,84792.0
169598,1,9,0,0,2025-07-01,256056.0,1477.0,1007.0,25.0
...,...,...,...,...,...,...,...,...,...
21840,3,10,2,3,2004-01-01,119289.0,3287.0,734203.0,15061.0
176680,2,10,1,0,2004-01-01,27814.0,708.0,13961.0,1781.0
60480,1,8,2,2,2004-01-01,156164.0,729.0,14643.0,23.0
22120,3,10,1,3,2004-01-01,58599.0,1641.0,494208.0,5545.0


If everything was done right, the number of rows should be equal to number of months from 01/2004 to 07/2025 * len(territorio_code.unique) * len(aeropuerto_escala_code.unique) * (movimiento_aeronave_code.unique) * (servicio_aereo_code.unique)

In [259]:
((12*20) + 7) * (len(df_f['TERRITORIO_CODE'].unique())) * (len(df_f['AEROPUERTO_ESCALA_CODE'].unique())) * \
(len(df_f['MOVIMIENTO_AERONAVE_CODE'].unique())) * (len(df_f['SERVICIO_AEREO_CODE'].unique()))

118560

In [234]:
df_f.rename({
'TERRITORIO_CODE': 'IslandId',
'AEROPUERTO_ESCALA_CODE': 'StopoverTerritoryId',
'MOVIMIENTO_AERONAVE_CODE': 'AircraftMovementId',
'SERVICIO_AEREO_CODE': 'AirServiceId',
'TIME_PERIOD#en': 'Month',
'OBS_VALUE': 'Passengers',
'OBS_VALUE_op': 'Operations',
'OBS_VALUE_goods': 'Goods',
'OBS_VALUE_mail': 'Mail'
}, inplace=True, axis=1)